# Midterm Project: Stroke Prediction 🧠

## 1. Problem Description
A stroke is a medical emergency that occurs when blood flow to the brain is interrupted. Early identification of individuals at high risk for a stroke allows healthcare providers to implement preventive measures, ultimately saving lives and reducing medical costs.

The goal of this project is to build a **Binary Classification** machine learning model that predicts whether a patient is likely to suffer a stroke (`stroke = 1`) or not (`stroke = 0`) based on their demographic and health records (e.g., age, average glucose level, BMI, hypertension, and smoking status).

## 2. Dataset
We are using a modified version of the **Healthcare Dataset - Stroke Data** from Kaggle.
* **Target Variable:** `stroke` (1 = stroke, 0 = no stroke).
* **Problem Type:** Binary Classification.

Let's begin by importing the necessary libraries, loading the data, and performing initial data cleaning (simulating missing values and outliers).

In [143]:
import pandas as pd
import numpy as np



# Read data

In [144]:
df=pd.read_csv("healthcare-dataset-stroke-data_manipulated.csv")
df.head()


,id,gender,age,hypertension,heart disease,ever married,work type,Residence type,avg glucose level,bmi,smoking status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
2,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
3,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
4,56669,Male,81.0,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,1


#  Clean column names (convert them to lowercase letters and replace spaces with underscores

In [145]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [146]:
df.columns = df.columns.str.replace('__', '_')

In [147]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
2,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
3,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
4,56669,Male,81.0,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,1


#  Clean text columns (categorical) to ensure there are no conflicting uppercase or lowercase letters

In [148]:
categorical_columns=list(df.select_dtypes(include=['object', 'str']).columns)
for col in categorical_columns:
    df[col]=df[col].astype(str).str.lower().str.replace(" ","_")
    

#  Dealing with missing values (Missing Values)

In [185]:
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['int64', 'float64']:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

In [150]:
print(df.isnull().sum())    
print(df.duplicated().sum())

id                   0
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64
0


#  Handling outliers (Outliers) in a BMI column

In [151]:
if 'bmi' in df.columns:
    df['bmi']=df['bmi'].clip(upper=60)

cols_to_describe = ['age', 'avg_glucose_level', 'bmi']
print("Dataset Shape:", df.shape)
print("\nNumerical Summary after handling outliers:\n", df[cols_to_describe].describe().round(2))
#print("\nMissing values after cleaning:\n", df.isnull().sum().sum())
#print("\nNumerical Summary after handling outliers:\n", df[['age', 'avg__glucose__level', 'bmi']].describe())

Dataset Shape: (4909, 12)

Numerical Summary after handling outliers:
            age  avg_glucose_level      bmi
count  4909.00            4909.00  4909.00
mean     42.87             105.31    28.87
std      22.56              44.42     7.71
min       0.08              55.12    10.30
25%      25.00              77.07    23.50
50%      44.00              91.68    28.10
75%      60.00             113.57    33.10
max      82.00             271.74    60.00


# Split Data (60% Train, 20% Val, 20% Test)

In [152]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score



In [153]:
df_full_train, df_test=train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val=train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train=df_train.reset_index(drop=True)
df_val=df_val.reset_index(drop=True)
df_test=df_test.reset_index(drop=True)

y_train = df_train['stroke'].values
y_val = df_val['stroke'].values
y_test = df_test['stroke'].values


del df_train['stroke']
del df_val['stroke']
del df_test['stroke']


print("Train shape:", df_train.shape)

print("--- Mutual Information (Categorical vs Stroke) ---")

Train shape: (2945, 11)
--- Mutual Information (Categorical vs Stroke) ---


In [154]:
def calculate_mi(series):
    return mutual_info_score(series, y_train)

In [155]:
mi_scores = df_train[categorical_columns].apply(calculate_mi).sort_values(ascending=False)
print(mi_scores.round(4))

ever_married      0.0239
work_type         0.0181
smoking_status    0.0124
gender            0.0018
residence_type    0.0000
dtype: float64


In [156]:
print("\n--- Correlation (Numerical vs Stroke) ---")



--- Correlation (Numerical vs Stroke) ---


# Numerical Features Correlation

In [157]:
num_cols = list(df.select_dtypes(include=['number']).columns)
num_features = [col for col in num_cols if col not in ['id', 'stroke']]
df_train_temp = df_train.copy()
df_train_temp['stroke'] = y_train

corr = df_train_temp[num_features + ['stroke']].corr()['stroke'].drop('stroke').sort_values(ascending=False)
print(corr.round(4))

avg_glucose_level    0.7142
hypertension         0.4481
age                  0.3624
bmi                  0.1965
heart_disease        0.1757
Name: stroke, dtype: float64


In [158]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# 1. تحديد خصائص التدريب الفئوية والرقمية (بناءً على الأعمدة المتوفرة)
 

  

  
 
# 2. تحويل البيانات لقاموس ثم تشفيرها باستخدام DictVectorizer
dv = DictVectorizer(sparse=False)
train_dict = df_train[features].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[features].to_dict(orient='records')
X_val = dv.transform(val_dict)

# 3. تدريب نموذج الانحدار اللوجستي (Baseline Model)
model_lr = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model_lr.fit(X_train, y_train)

# 4. التنبؤ وحساب مقياس ROC AUC على مجموعة التحقق
y_pred_proba = model_lr.predict_proba(X_val)[:, 1]
auc_lr = roc_auc_score(y_val, y_pred_proba)

print("تم تجهيز التدريب بنجاح!")
print(f"X_train shape: {X_train.shape}")
print(f"Logistic Regression Validation ROC AUC: {auc_lr:.4f}")

تم تجهيز التدريب بنجاح!
X_train shape: (2945, 21)
Logistic Regression Validation ROC AUC: 0.9767


In [159]:
from sklearn.feature_extraction import DictVectorizer

numerical = [
    'age',
    'hypertension',
    'heart_disease',
    'avg_glucose_level',
    'bmi'
]

categorical = [
    'gender',
    'ever_married',
    'work_type',
    'residence_type',
    'smoking_status'
]

features = categorical + numerical

dv = DictVectorizer(sparse=False)

train_dict = df_train[features].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[features].to_dict(orient='records')
X_val = dv.transform(val_dict)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Number of features:", len(dv.get_feature_names_out()))

X_train: (2945, 21)
X_val: (982, 21)
Number of features: 21


# Logistic Regression.

In [160]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

model_lr = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
)

# Train the model
model_lr.fit(X_train, y_train)

# Predict probabilities on validation data
y_pred_proba = model_lr.predict_proba(X_val)[:, 1]

# Evaluate using ROC AUC
auc_lr = roc_auc_score(y_val, y_pred_proba)

print(f"Logistic Regression Validation ROC AUC: {auc_lr:.4f}")

Logistic Regression Validation ROC AUC: 0.9767


In [161]:
print("--- Target Distribution ---")
print(df['stroke'].value_counts())
print("\nTarget Percentages:")
print(df['stroke'].value_counts(normalize=True).round(4))

print("\n--- Stroke Rate by Feature ---")
for col in ['age', 'avg_glucose_level', 'bmi',
            'hypertension', 'heart_disease']:
    print(f"\n{col}:")
    print(df.groupby('stroke')[col].mean().round(3))

print("\n--- Feature Columns ---")
print(dv.get_feature_names_out())

print("\n--- Target Leakage Check ---")
print("stroke in features:", 'stroke' in features)
print("id in features:", 'id' in features)

--- Target Distribution ---
stroke
0    3862
1    1047
Name: count, dtype: int64

Target Percentages:
stroke
0    0.7867
1    0.2133
Name: proportion, dtype: float64

--- Stroke Rate by Feature ---

age:
stroke
0    38.601
1    58.593
Name: age, dtype: float64

avg_glucose_level:
stroke
0     88.737
1    166.421
Name: avg_glucose_level, dtype: float64

bmi:
stroke
0    28.064
1    31.834
Name: bmi, dtype: float64

hypertension:
stroke
0    0.024
1    0.342
Name: hypertension, dtype: float64

heart_disease:
stroke
0    0.028
1    0.130
Name: heart_disease, dtype: float64

--- Feature Columns ---
['age' 'avg_glucose_level' 'bmi' 'ever_married=no' 'ever_married=yes'
 'gender=female' 'gender=male' 'gender=other' 'heart_disease'
 'hypertension' 'residence_type=rural' 'residence_type=urban'
 'smoking_status=formerly_smoked' 'smoking_status=never_smoked'
 'smoking_status=smokes' 'smoking_status=unknown' 'work_type=children'
 'work_type=govt__job' 'work_type=never__worked' 'work_type=private'


In [162]:
for c in [0.000001, 0.00001, 0.0001, 0.001,
          0.01, 0.1, 1, 10, 100]:

    model = LogisticRegression(
        solver='liblinear',
        C=c,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]

    auc = roc_auc_score(y_val, y_pred)

    print(f"C={c}: Validation AUC={auc:.4f}")

C=1e-06: Validation AUC=0.0888
C=1e-05: Validation AUC=0.6607
C=0.0001: Validation AUC=0.7791
C=0.001: Validation AUC=0.8170
C=0.01: Validation AUC=0.9203
C=0.1: Validation AUC=0.9747
C=1: Validation AUC=0.9767
C=10: Validation AUC=0.9767
C=100: Validation AUC=0.9767


In [163]:
model_lr = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model_lr.fit(X_train, y_train)

# 2. التنبؤ بالاحتمالات على بيانات التحقق (Validation Set)
y_pred_proba = model_lr.predict_proba(X_val)[:, 1]

# 3. حساب مقياس التقييم ROC AUC
auc_lr = roc_auc_score(y_val, y_pred_proba)
print(f"Logistic Regression Validation ROC AUC: {auc_lr:.4f}")

Logistic Regression Validation ROC AUC: 0.9767


# Hyperparameter Tuning

# Decision Tree

In [164]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

model_dt = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=10,
    random_state=42
)

model_dt.fit(X_train, y_train)

y_pred_proba_dt = model_dt.predict_proba(X_val)[:, 1]

auc_dt = roc_auc_score(y_val, y_pred_proba_dt)

print(f"Decision Tree Validation ROC AUC: {auc_dt:.4f}")

Decision Tree Validation ROC AUC: 0.9878


# Decision Tree Hyperparameter Tuning

In [165]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

for depth in [3, 5, 7, 10,  None]:
    for leaf in [1, 5, 10, 20, 50]:

        model = DecisionTreeClassifier(
            max_depth=depth,
            min_samples_leaf=leaf,
            random_state=42
        )

        model.fit(X_train, y_train)

        y_pred = model.predict_proba(X_val)[:, 1]

        auc = roc_auc_score(y_val, y_pred)

        print(
            f"depth={depth}, leaf={leaf}: "
            f"Validation AUC={auc:.4f}"
        )

depth=3, leaf=1: Validation AUC=0.9777
depth=3, leaf=5: Validation AUC=0.9777
depth=3, leaf=10: Validation AUC=0.9777
depth=3, leaf=20: Validation AUC=0.9777
depth=3, leaf=50: Validation AUC=0.9770
depth=5, leaf=1: Validation AUC=0.9882
depth=5, leaf=5: Validation AUC=0.9882
depth=5, leaf=10: Validation AUC=0.9878
depth=5, leaf=20: Validation AUC=0.9875
depth=5, leaf=50: Validation AUC=0.9878
depth=7, leaf=1: Validation AUC=0.9889
depth=7, leaf=5: Validation AUC=0.9878
depth=7, leaf=10: Validation AUC=0.9914
depth=7, leaf=20: Validation AUC=0.9880
depth=7, leaf=50: Validation AUC=0.9884
depth=10, leaf=1: Validation AUC=0.9816
depth=10, leaf=5: Validation AUC=0.9869
depth=10, leaf=10: Validation AUC=0.9874
depth=10, leaf=20: Validation AUC=0.9842
depth=10, leaf=50: Validation AUC=0.9842
depth=None, leaf=1: Validation AUC=0.9556
depth=None, leaf=5: Validation AUC=0.9737
depth=None, leaf=10: Validation AUC=0.9822
depth=None, leaf=20: Validation AUC=0.9778
depth=None, leaf=50: Validation A

In [166]:
DecisionTreeClassifier(
    max_depth=7,
    min_samples_leaf=10,
    random_state=42
)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",7
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples i

In [167]:
best_dt = DecisionTreeClassifier(
    max_depth=7,
    min_samples_leaf=10,
    random_state=42
)

best_dt.fit(X_train, y_train)

y_pred_best_dt = best_dt.predict_proba(X_val)[:, 1]

print(
    "Best Decision Tree AUC:",
    round(roc_auc_score(y_val, y_pred_best_dt), 4)
)

Best Decision Tree AUC: 0.9914


In [168]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

model_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=7,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

model_rf.fit(X_train, y_train)

y_pred_proba_rf = model_rf.predict_proba(X_val)[:, 1]

auc_rf = roc_auc_score(y_val, y_pred_proba_rf)

print(f"Random Forest Validation ROC AUC: {auc_rf:.4f}")

Random Forest Validation ROC AUC: 0.9870


# Random Forest Tuning

In [169]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

for n in [100, 200, 300]:
    for depth in [5, 7, 10, None]:
        for leaf in [5, 10, 20]:

            model = RandomForestClassifier(
                n_estimators=n,
                max_depth=depth,
                min_samples_leaf=leaf,
                random_state=42,
                n_jobs=-1
            )

            model.fit(X_train, y_train)

            y_pred = model.predict_proba(X_val)[:, 1]

            auc = roc_auc_score(y_val, y_pred)

            print(
                f"n={n}, depth={depth}, leaf={leaf}: "
                f"AUC={auc:.4f}"
            )

n=100, depth=5, leaf=5: AUC=0.9873
n=100, depth=5, leaf=10: AUC=0.9876
n=100, depth=5, leaf=20: AUC=0.9866
n=100, depth=7, leaf=5: AUC=0.9879
n=100, depth=7, leaf=10: AUC=0.9878
n=100, depth=7, leaf=20: AUC=0.9855
n=100, depth=10, leaf=5: AUC=0.9845
n=100, depth=10, leaf=10: AUC=0.9856
n=100, depth=10, leaf=20: AUC=0.9861
n=100, depth=None, leaf=5: AUC=0.9822
n=100, depth=None, leaf=10: AUC=0.9843
n=100, depth=None, leaf=20: AUC=0.9859
n=200, depth=5, leaf=5: AUC=0.9879
n=200, depth=5, leaf=10: AUC=0.9878
n=200, depth=5, leaf=20: AUC=0.9874
n=200, depth=7, leaf=5: AUC=0.9877
n=200, depth=7, leaf=10: AUC=0.9870
n=200, depth=7, leaf=20: AUC=0.9867
n=200, depth=10, leaf=5: AUC=0.9849
n=200, depth=10, leaf=10: AUC=0.9850
n=200, depth=10, leaf=20: AUC=0.9859
n=200, depth=None, leaf=5: AUC=0.9825
n=200, depth=None, leaf=10: AUC=0.9843
n=200, depth=None, leaf=20: AUC=0.9855
n=300, depth=5, leaf=5: AUC=0.9882
n=300, depth=5, leaf=10: AUC=0.9881
n=300, depth=5, leaf=20: AUC=0.9878
n=300, depth=

# Best settings shown in experiments:

In [170]:
RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total nu

#   Install Best Random Forest

In [171]:
best_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X_train, y_train)

y_pred_best_rf = best_rf.predict_proba(X_val)[:, 1]

print(
    "Best Random Forest AUC:",
    round(roc_auc_score(y_val, y_pred_best_rf), 4)
)

Best Random Forest AUC: 0.9882


# XGBoost

In [172]:
import xgboost as xgb

print(xgb.__version__)

3.4.1


In [173]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

model_xgb = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

model_xgb.fit(X_train, y_train)

y_pred_proba_xgb = model_xgb.predict_proba(X_val)[:, 1]

auc_xgb = roc_auc_score(y_val, y_pred_proba_xgb)

print(f"XGBoost Validation ROC AUC: {auc_xgb:.4f}")

XGBoost Validation ROC AUC: 0.9892


# XGBoost Hyperparameter Tuning

In [174]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

for n in [100, 200, 300]:
    for depth in [2, 3, 4, 5]:
        for lr in [0.01, 0.05, 0.1]:

            model = XGBClassifier(
                n_estimators=n,
                max_depth=depth,
                learning_rate=lr,
                objective='binary:logistic',
                eval_metric='logloss',
                random_state=42,
                n_jobs=-1
            )

            model.fit(X_train, y_train)

            y_pred = model.predict_proba(X_val)[:, 1]

            auc = roc_auc_score(y_val, y_pred)

            print(
                f"n={n}, depth={depth}, lr={lr}: "
                f"Validation AUC={auc:.4f}"
            )

n=100, depth=2, lr=0.01: Validation AUC=0.9831
n=100, depth=2, lr=0.05: Validation AUC=0.9880
n=100, depth=2, lr=0.1: Validation AUC=0.9893
n=100, depth=3, lr=0.01: Validation AUC=0.9798
n=100, depth=3, lr=0.05: Validation AUC=0.9887
n=100, depth=3, lr=0.1: Validation AUC=0.9892
n=100, depth=4, lr=0.01: Validation AUC=0.9873
n=100, depth=4, lr=0.05: Validation AUC=0.9887
n=100, depth=4, lr=0.1: Validation AUC=0.9888
n=100, depth=5, lr=0.01: Validation AUC=0.9882
n=100, depth=5, lr=0.05: Validation AUC=0.9881
n=100, depth=5, lr=0.1: Validation AUC=0.9882
n=200, depth=2, lr=0.01: Validation AUC=0.9848
n=200, depth=2, lr=0.05: Validation AUC=0.9889
n=200, depth=2, lr=0.1: Validation AUC=0.9894
n=200, depth=3, lr=0.01: Validation AUC=0.9879
n=200, depth=3, lr=0.05: Validation AUC=0.9891
n=200, depth=3, lr=0.1: Validation AUC=0.9893
n=200, depth=4, lr=0.01: Validation AUC=0.9886
n=200, depth=4, lr=0.05: Validation AUC=0.9889
n=200, depth=4, lr=0.1: Validation AUC=0.9871
n=200, depth=5, lr=0

# n_estimators	max_depth	learning_rate	AUC
200	2	0.1	0.9894
300	3	0.05	0.9894

In [175]:
best_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=2,
    learning_rate=0.1,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

best_xgb.fit(X_train, y_train)

y_pred_best_xgb = best_xgb.predict_proba(X_val)[:, 1]

print(
    "Best XGBoost AUC:",
    round(roc_auc_score(y_val, y_pred_best_xgb), 4)
)

Best XGBoost AUC: 0.9894


# Overfitting Check and Evaluation

In [176]:
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

# Predictions
y_train_pred = best_xgb.predict_proba(X_train)[:, 1]
y_val_pred = best_xgb.predict_proba(X_val)[:, 1]

# ROC AUC
print("Training AUC:", round(
    roc_auc_score(y_train, y_train_pred), 4
))

print("Validation AUC:", round(
    roc_auc_score(y_val, y_val_pred), 4
))

# Classification at threshold = 0.5
y_val_class = (y_val_pred >= 0.5).astype(int)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_class))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_val_class,
    zero_division=0
))

Training AUC: 0.9904
Validation AUC: 0.9894

Confusion Matrix:
[[771   2]
 [ 16 193]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       773
           1       0.99      0.92      0.96       209

    accuracy                           0.98       982
   macro avg       0.98      0.96      0.97       982
weighted avg       0.98      0.98      0.98       982



# Threshold Tuning

In [177]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5,
              0.6, 0.7, 0.8, 0.9]

print("Threshold | Precision | Recall | F1")

for threshold in thresholds:

    y_pred = (y_val_pred >= threshold).astype(int)

    precision = precision_score(
        y_val, y_pred, zero_division=0
    )

    recall = recall_score(
        y_val, y_pred, zero_division=0
    )

    f1 = f1_score(
        y_val, y_pred, zero_division=0
    )

    print(
        f"{threshold:9.1f} | "
        f"{precision:.4f} | "
        f"{recall:.4f} | "
        f"{f1:.4f}"
    )

Threshold | Precision | Recall | F1
      0.1 | 0.7835 | 0.9522 | 0.8596
      0.2 | 0.9512 | 0.9330 | 0.9420
      0.3 | 0.9798 | 0.9282 | 0.9533
      0.4 | 0.9848 | 0.9282 | 0.9557
      0.5 | 0.9897 | 0.9234 | 0.9554
      0.6 | 0.9948 | 0.9234 | 0.9578
      0.7 | 1.0000 | 0.9139 | 0.9550
      0.8 | 1.0000 | 0.8995 | 0.9471
      0.9 | 1.0000 | 0.8517 | 0.9199


# compare Decision Tree with XGBoost fairly

In [178]:
from sklearn.metrics import classification_report

models = {
    "Decision Tree": best_dt,
    "Random Forest": best_rf,
    "XGBoost": best_xgb
}

for name, model in models.items():

    predictions = model.predict_proba(X_val)[:, 1]

    y_class = (predictions >= 0.5).astype(int)

    print(f"\n===== {name} =====")

    print(
        "ROC AUC:",
        round(roc_auc_score(y_val, predictions), 4)
    )

    print(
        classification_report(
            y_val,
            y_class,
            zero_division=0
        )
    )


===== Decision Tree =====
ROC AUC: 0.9914
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       773
           1       0.99      0.94      0.97       209

    accuracy                           0.99       982
   macro avg       0.99      0.97      0.98       982
weighted avg       0.99      0.99      0.99       982


===== Random Forest =====
ROC AUC: 0.9882
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       773
           1       1.00      0.93      0.96       209

    accuracy                           0.98       982
   macro avg       0.99      0.96      0.98       982
weighted avg       0.99      0.98      0.98       982


===== XGBoost =====
ROC AUC: 0.9894
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       773
           1       0.99      0.92      0.96       209

    accuracy                           0.98       982
   ma

# I suggest adopting Decision Tree as the first candidate, with the following settings:

In [182]:
best_dt = DecisionTreeClassifier(
    max_depth=7,
    min_samples_leaf=10,
    random_state=42
)

In [183]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Create the best Decision Tree
best_dt = DecisionTreeClassifier(
    max_depth=7,
    min_samples_leaf=10,
    random_state=42
)

# 2. Train the model
best_dt.fit(X_train, y_train)

# 3. Predict probabilities
y_val_pred_dt = best_dt.predict_proba(X_val)[:, 1]

# 4. Check AUC
print(
    "Decision Tree AUC:",
    round(roc_auc_score(y_val, y_val_pred_dt), 4)
)

# 5. Threshold tuning
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5,
              0.6, 0.7, 0.8, 0.9]

print("\nThreshold | Precision | Recall | F1")

for threshold in thresholds:

    y_pred = (y_val_pred_dt >= threshold).astype(int)

    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)

    print(
        f"{threshold:9.1f} | "
        f"{precision:.4f} | "
        f"{recall:.4f} | "
        f"{f1:.4f}"
    )

Decision Tree AUC: 0.9914

Threshold | Precision | Recall | F1
      0.1 | 0.7710 | 0.9665 | 0.8577
      0.2 | 0.9289 | 0.9378 | 0.9333
      0.3 | 0.9949 | 0.9378 | 0.9655
      0.4 | 0.9949 | 0.9378 | 0.9655
      0.5 | 0.9949 | 0.9378 | 0.9655
      0.6 | 1.0000 | 0.9330 | 0.9653
      0.7 | 1.0000 | 0.9330 | 0.9653
      0.8 | 1.0000 | 0.9330 | 0.9653
      0.9 | 1.0000 | 0.9330 | 0.9653


In [184]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_val_pred_dt = best_dt.predict_proba(X_val)[:, 1]

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5,
              0.6, 0.7, 0.8, 0.9]

for threshold in thresholds:

    y_pred = (y_val_pred_dt >= threshold).astype(int)

    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)

    cm = confusion_matrix(y_val, y_pred)

    print(f"\nThreshold: {threshold}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1: {f1:.4f}")
    print("Confusion Matrix:")
    print(cm)


Threshold: 0.1
Precision: 0.7710
Recall: 0.9665
F1: 0.8577
Confusion Matrix:
[[713  60]
 [  7 202]]

Threshold: 0.2
Precision: 0.9289
Recall: 0.9378
F1: 0.9333
Confusion Matrix:
[[758  15]
 [ 13 196]]

Threshold: 0.3
Precision: 0.9949
Recall: 0.9378
F1: 0.9655
Confusion Matrix:
[[772   1]
 [ 13 196]]

Threshold: 0.4
Precision: 0.9949
Recall: 0.9378
F1: 0.9655
Confusion Matrix:
[[772   1]
 [ 13 196]]

Threshold: 0.5
Precision: 0.9949
Recall: 0.9378
F1: 0.9655
Confusion Matrix:
[[772   1]
 [ 13 196]]

Threshold: 0.6
Precision: 1.0000
Recall: 0.9330
F1: 0.9653
Confusion Matrix:
[[773   0]
 [ 14 195]]

Threshold: 0.7
Precision: 1.0000
Recall: 0.9330
F1: 0.9653
Confusion Matrix:
[[773   0]
 [ 14 195]]

Threshold: 0.8
Precision: 1.0000
Recall: 0.9330
F1: 0.9653
Confusion Matrix:
[[773   0]
 [ 14 195]]

Threshold: 0.9
Precision: 1.0000
Recall: 0.9330
F1: 0.9653
Confusion Matrix:
[[773   0]
 [ 14 195]]


In [186]:
print("Missing values:")
print(df.isnull().sum())

print("\nDataset shape:")
print(df.shape)

print("\nSplit shapes:")
print("Train:", df_train.shape)
print("Validation:", df_val.shape)
print("Test:", df_test.shape)

Missing values:
id                   0
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64

Dataset shape:
(4909, 12)

Split shapes:
Train: (2945, 11)
Validation: (982, 11)
Test: (982, 11)


In [187]:
print("BMI statistics:")
print(df['bmi'].describe())

print("\nBMI values above 60:")
print((df['bmi'] > 60).sum())

print("\nBMI maximum:")
print(df['bmi'].max())

BMI statistics:
count    4909.000000
mean       28.868059
std         7.714013
min        10.300000
25%        23.500000
50%        28.100000
75%        33.100000
max        60.000000
Name: bmi, dtype: float64

BMI values above 60:
0

BMI maximum:
60.0


In [188]:
from sklearn.model_selection import train_test_split

# Split into training and test sets
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)

# Split training into training and validation sets
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

# Reset indexes
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print("Train:", df_train.shape)
print("Validation:", df_val.shape)
print("Test:", df_test.shape)

Train: (2945, 12)
Validation: (982, 12)
Test: (982, 12)


In [189]:
# Identify numerical and categorical columns
numerical_cols = df_train.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

categorical_cols = df_train.select_dtypes(
    include=['object', 'str']
).columns.tolist()

# Exclude target and ID
numerical_cols.remove('stroke')
numerical_cols.remove('id')

# Calculate imputation values using training data only
numerical_medians = df_train[numerical_cols].median()
categorical_modes = df_train[categorical_cols].mode().iloc[0]

print("Numerical medians:")
print(numerical_medians)

print("\nCategorical modes:")
print(categorical_modes)

Numerical medians:
age                  44.00
hypertension          0.00
heart_disease         0.00
avg_glucose_level    91.45
bmi                  28.00
dtype: float64

Categorical modes:
gender                  female
ever_married               yes
work_type              private
residence_type           rural
smoking_status    never_smoked
Name: 0, dtype: str
